# BC Training — Kaggle

Запускает supervised обучение политики (ViT) на собранном датасете.

**Перед запуском:**
1. Добавь датасет с файлами `bc_dataset.npz`, `bc_dataset_obs.dat`, `bc_dataset_act.dat` через Add Input
2. Accelerator: T4 x2
3. Internet: On

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден — включи T4 в настройках!'}")

In [ ]:
# Клонируем репо и устанавливаем зависимости
# torch не указываем — на Kaggle уже установлен
!git clone https://github.com/Andrew82mm/RL_practice.git /kaggle/working/RL_practice
%cd /kaggle/working/RL_practice
!pip install -q sb3-contrib gymnasium scipy tqdm

In [ ]:
import os, shutil, numpy as np

# Находим датасет через os.walk (Kaggle меняет '_' на '-' в именах папок)
LOCAL_DIR = "/kaggle/working/dataset"
os.makedirs(LOCAL_DIR, exist_ok=True)

def _find_dataset_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "bc_dataset.npz" in files:
            return root
    return None

KAGGLE_DATASET_DIR = _find_dataset_dir()
if KAGGLE_DATASET_DIR is None:
    print("Содержимое /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(f"  {root}: {files}")
    raise FileNotFoundError("Датасет не найден! Проверь что он добавлен через Add Input")
print(f"Датасет найден: {KAGGLE_DATASET_DIR}")

# Копируем на локальный диск (важно для скорости случайных чтений DataLoader)
print("Копируем данные (~7.2 ГБ)...")
for fname in ["bc_dataset.npz", "bc_dataset_obs.dat", "bc_dataset_act.dat"]:
    dst = f"{LOCAL_DIR}/{fname}"
    if not os.path.exists(dst):
        shutil.copy2(f"{KAGGLE_DATASET_DIR}/{fname}", dst)
    print(f"  {fname}: {os.path.getsize(dst)/1024**2:.0f} MB — OK")

# Патчим пути в .npz
meta = np.load(f"{LOCAL_DIR}/bc_dataset.npz", allow_pickle=True)
np.savez(f"{LOCAL_DIR}/bc_dataset.npz",
    n_samples=meta["n_samples"],
    obs_shape=meta["obs_shape"],
    obs_path=np.array(f"{LOCAL_DIR}/bc_dataset_obs.dat"),
    act_path=np.array(f"{LOCAL_DIR}/bc_dataset_act.dat"),
)
print(f"\nДатасет готов: {int(meta['n_samples']):,} пар")

In [ ]:
%cd /kaggle/working/RL_practice

# Запускаем BC обучение ViT
# Модель сохраняется в /kaggle/working/ — скачай через Output после завершения
!python training/bc_train.py \
    --only-train \
    --arch vit \
    --dataset-path "/kaggle/working/dataset/bc_dataset.npz" \
    --pretrained-path "/kaggle/working/vit_bc_pretrained"

In [ ]:
# Проверяем результат
import os
model_path = "/kaggle/working/vit_bc_pretrained.zip"
print(f"Модель: {os.path.getsize(model_path)/1024**2:.1f} MB")
print("Скачай vit_bc_pretrained.zip через вкладку Output →")